# 01 — ENSO (BoM relative Niño3.4) and IOD classification from OISST

**Relative Niño3.4** (BoM, adopted Sep 2025):
RN34 = S × [(N34 − N34clim) − (TropMean − TropMeanclim)], TropMean = 20°S–20°N mean SST,
anomalies vs `cfg.INDEX_BASELINE` (1991–2020), S rescales RN34 to the variance of the traditional
index (over `cfg.SCALE_PERIOD`). Computed here from OISST.

- An El Niño (La Niña) **event** = ≥ `ENSO_MIN_MONTHS` consecutive months with RN34 ≥ +`ENSO_THRESH`
  (≤ −`ENSO_THRESH`); BoM: sustained monthly values beyond ±0.8 °C. "Sustained" is not quantified by
  BoM — `ENSO_MIN_MONTHS` is our choice.
- **ENSO year** Y = Jun(Y)–May(Y+1); phase from the event (if any) covering Dec(Y)–Feb(Y+1).
  **Strong** El Niño: event peak ≥ `STRONG_THRESH` (our choice; BoM defines no strength classes).
- IOD flag from the SON(Y) mean DMI vs `IOD_THRESH`.
- The traditional Niño3.4 (3-month mean, same baseline) is kept in the table for comparison.

Needs the `trop` box in `index_boxes_daily.nc` — rerun 00 (or just its index-box cell) if missing.
Output: `enso_years.csv`, `indices_monthly.nc`.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import ausenso as cfg
warnings.filterwarnings("ignore", category=RuntimeWarning)
cfg.ensure_dirs()
print(dict(index_baseline=cfg.INDEX_BASELINE, scale_period=cfg.SCALE_PERIOD, smooth=cfg.ENSO_SMOOTH,
           thresh=cfg.ENSO_THRESH, min_months=cfg.ENSO_MIN_MONTHS, strong=cfg.STRONG_THRESH,
           dmi_detrend=cfg.INDEX_DETREND, iod=cfg.IOD_THRESH))

In [ ]:
daily = xr.open_dataset(cfg.F_INDICES_DAILY)
assert "trop" in daily, "index_boxes_daily.nc has no 'trop' box — rerun the index-box cell of 00"
mon = daily.resample(time="MS").mean()
ndays = daily.nino34.resample(time="MS").count()
mon = mon.where(ndays == ndays.time.dt.days_in_month, drop=True)   # complete months only
print(str(mon.time.values[0])[:7], "→", str(mon.time.values[-1])[:7])

In [ ]:
base = mon.sel(time=slice(str(cfg.INDEX_BASELINE[0]), str(cfg.INDEX_BASELINE[1])))
anom = (mon.groupby("time.month") - base.groupby("time.month").mean()).drop_vars("month")

n34_trad = anom.nino34
raw = anom.nino34 - anom.trop
sp = slice(str(cfg.SCALE_PERIOD[0]), str(cfg.SCALE_PERIOD[1]))
S = float(n34_trad.sel(time=sp).std() / raw.sel(time=sp).std())
rn34 = S * raw
print(f"S = {S:.3f}  |  corr(RN34, N34) = {float(xr.corr(rn34, n34_trad)):.3f}")

dmi = anom.dmi_w - anom.dmi_e
if cfg.INDEX_DETREND:
    fit = dmi.polyfit("time", deg=1)
    trend = xr.polyval(dmi.time, fit.polyfit_coefficients)
    dmi = dmi - (trend - trend.mean())
if cfg.ENSO_SMOOTH > 1:
    rn34_c = rn34.rolling(time=cfg.ENSO_SMOOTH, center=True).mean()
else:
    rn34_c = rn34
n34_3 = n34_trad.rolling(time=3, center=True).mean()

In [ ]:
def runs(mask):
    """(start, end) index pairs of True runs."""
    m = np.r_[False, np.asarray(mask), False].astype(int)
    d = np.diff(m)
    return list(zip(np.where(d == 1)[0], np.where(d == -1)[0] - 1))

v = rn34_c.values
events = []
for sign, ph in [(1, "EN"), (-1, "LN")]:
    for s, e in runs(np.nan_to_num(sign * v) >= cfg.ENSO_THRESH):
        if e - s + 1 >= cfg.ENSO_MIN_MONTHS:
            seg = v[s:e + 1]
            j = int(np.argmax(sign * seg))
            events.append(dict(phase=ph, start=rn34_c.time.values[s], end=rn34_c.time.values[e],
                               n_months=e - s + 1, peak=round(float(seg[j]), 2),
                               peak_time=rn34_c.time.values[s + j]))
events = pd.DataFrame(events).sort_values("start").reset_index(drop=True)
events

In [ ]:
rows = []
for y in range(cfg.ENSO_YEARS[0], cfg.ENSO_YEARS[1] + 1):
    d0, d1 = np.datetime64(f"{y}-12-01"), np.datetime64(f"{y + 1}-02-01")
    ev = events[(events.start <= d1) & (events.end >= d0)]
    phase, peak = (ev.iloc[0].phase, ev.iloc[0].peak) if len(ev) else ("N", np.nan)
    djf = slice(f"{y}-12-01", f"{y + 1}-02-28")
    son = dmi.sel(time=slice(f"{y}-09-01", f"{y}-11-30")).mean().item()
    rows.append(dict(enso_year=y, label=f"{y}/{str(y + 1)[-2:]}",
                     rn34_DJF=round(float(rn34.sel(time=djf).mean()), 2),
                     n34_DJF=round(float(n34_trad.sel(time=djf).mean()), 2),
                     phase=phase, peak=peak,
                     strong_EN=bool(phase == "EN" and peak >= cfg.STRONG_THRESH),
                     dmi_SON=round(son, 2),
                     iod="+" if son >= cfg.IOD_THRESH else ("-" if son <= -cfg.IOD_THRESH else "0")))
table = pd.DataFrame(rows).set_index("enso_year")
table.to_csv(cfg.F_ENSO_TABLE)
pd.set_option("display.max_rows", 60)
table

In [ ]:
for nm, yrs in cfg.phase_years(table).items():
    print(f"{nm:17s} n={len(yrs):2d}: {[table.label[y] for y in yrs]}")
print()
print("El Niño years with +IOD:", [table.label[y] for y in table.index[(table.phase == 'EN') & (table.iod == '+')]])
print("La Niña years with -IOD:", [table.label[y] for y in table.index[(table.phase == 'LN') & (table.iod == '-')]])

Check the event list and `peak` column before fixing the definitions: membership is sensitive to
`ENSO_MIN_MONTHS` and `STRONG_THRESH`, and this RN34 is built from OISST rather than BoM's own SST
product, so compare against BoM's published relative Niño3.4 for a few known events.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
ax[0].fill_between(rn34.time, 0, rn34, where=rn34 > 0, color="C3", lw=0, step="mid")
ax[0].fill_between(rn34.time, 0, rn34, where=rn34 < 0, color="C0", lw=0, step="mid")
ax[0].plot(n34_3.time, n34_3, "k", lw=0.6, label="traditional N3.4 (3-mo)")
for th in (cfg.ENSO_THRESH, -cfg.ENSO_THRESH, cfg.STRONG_THRESH):
    ax[0].axhline(th, color="k", lw=0.5, ls=":")
for y in table.index[table.strong_EN]:
    ax[0].axvspan(pd.Timestamp(f"{y}-06-01"), pd.Timestamp(f"{y + 1}-05-31"), color="0.85", zorder=0)
ax[0].set_ylabel("Relative Niño3.4 (°C)"); ax[0].legend(loc="upper left", fontsize=8)
ax[1].plot(dmi.time, dmi.rolling(time=3, center=True).mean(), "k", lw=0.8)
ax[1].axhline(cfg.IOD_THRESH, color="k", lw=0.5, ls=":"); ax[1].axhline(-cfg.IOD_THRESH, color="k", lw=0.5, ls=":")
ax[1].set_ylabel("DMI (°C, 3-mo)")
fig.savefig(cfg.FIG / "01_indices.png", dpi=200, bbox_inches="tight")

In [ ]:
out = xr.Dataset({"rn34": rn34, "nino34": n34_trad, "nino34_3mo": n34_3, "trop": anom.trop, "dmi": dmi})
out.attrs.update(S=S, index_baseline=str(cfg.INDEX_BASELINE), scale_period=str(cfg.SCALE_PERIOD))
out.to_netcdf(cfg.F_INDICES_MONTHLY)